# 무료 로컬 RAG 확장 실습
API 키가 필요 없습니다. run_local_ai.bat로 모델을 켠 다음 위에서부터 실행하세요. 실행 전 상태의 노트북입니다. 직접 바꾼 설정과 출력은 저장해서 과제에 남기세요.

In [ ]:
from pathlib import Path
from dataclasses import replace
from advanced_rag import Options, StudyRag
from ingest import load_file, load_url
from local_ai import model_status, describe_image
ROOT = Path.cwd()
print("로컬 모델 준비:", model_status())

## 1. 내 문서 로드
study_wiki.md를 본인의 파일 경로로 바꿔보세요. 이미지/PDF 시각 분석은 vision=True로 바꿉니다. 기본 앞3페이지를 분석합니다.

In [ ]:
path = ROOT / "data/study_wiki.md"
docs = load_file(path.name, path.read_bytes(), vision=False, visual_pages=3, describe=describe_image)
print(len(docs), docs[0].metadata)
print(docs[0].page_content[:500])

## 2. 청킹과 로컬 임베딩·저장소
mode="local"은 무료 생성형 QA입니다. splitter는 recursive/character/semantic, store는 faiss/chroma, search는 similarity/bm25/hybrid/mmr/threshold/multiquery 중 선택합니다.

In [ ]:
settings = Options(mode="local", chunk_size=1000, overlap=100, k=4, search="hybrid")
bot = StudyRag(docs, settings, progress=print)
print("청크:", len(bot.chunks))

## 3. 첫 질문과 출처

In [ ]:
result = bot.ask("2주차 과제 제출물은 무엇인가요?", session_id="my-study")
print(result["answer"])
for source in result["sources"]:
    print(source["number"], source["source"], source["text"][:200])

## 4. 대화 기억과 검색 질문 재작성
같은 session_id를 사용해야 이전 대화를 참고합니다. 다른 ID는 별도 대화입니다.

In [ ]:
followup = bot.ask("그중 문서에 없는 질문도 포함해야 하나요?", session_id="my-study")
print("검색 질의:", followup["query"])
print(followup["answer"])

## 5. 검색 비교
동일한 인덱스로 비교합니다. 검색된 청크에 실제 답이 들어 있는지 직접 확인하세요.

In [ ]:
for strategy in ["similarity", "bm25", "hybrid", "mmr", "multiquery"]:
    found = bot.resolve(bot.retrieve("과제 제출물", strategy))
    print(strategy, [(d.metadata["source"], d.metadata["chunk_id"]) for d in found])

## 6. 선택 실험: RAPTOR와 5단계 밀도 요약
기본값 False입니다. 직접 True로 바꾸면 실행합니다. RAPTOR는120청크, 밀도 요약은본문12000자까지입니다. 시간이 더 걸립니다.

In [ ]:
RUN_ADVANCED = False
if RUN_ADVANCED:
    tree_bot = StudyRag(docs, replace(settings, raptor=True))
    print(tree_bot.ask("전체 문서의 주요 내용은?")["answer"])
    for index, summary in enumerate(bot.density_summary(), 1):
        print(index, summary)

## 7. 선택 실험: 웹과 이미지
직접 공개 웹 주소 또는 이미지 경로로 변경하세요. 모델과 문서는 외부 AI API에 보내지 않습니다. 웹 주소의 서버에는 자료를 내려받는 요청이 전달됩니다.

In [ ]:
RUN_EXTRA = False
if RUN_EXTRA:
    web_docs = load_url("https://example.com")
    print(web_docs[0].page_content)
    image_path = ROOT / "outputs/vision-test.png"
    visual_docs = load_file(image_path.name, image_path.read_bytes(), vision=True, describe=describe_image)
    visual_bot = StudyRag(visual_docs, settings)
    print(visual_bot.ask("이미지 내용을 설명해 주세요.")["answer"])

## 8. 구현 과정 기록
실제 실행한 결과만 저장합니다. docs/나의_구현_과정.md에 자료 선택 이유·수정한 코드·결과·오류를 작성하세요.

In [ ]:
import json
from datetime import datetime
out = ROOT / "outputs"
out.mkdir(exist_ok=True)
record = {"settings": settings.__dict__, "first_answer": result, "followup": followup}
path = out / ("local-study-" + datetime.now().strftime("%Y%m%d-%H%M%S") + ".json")
path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
print(path)